# KALE retrain on our train split (E4, starting point for E5)

Retrains KALE for a few epochs on our 2,271 training paintings (not KALE's full 2,524), scores
every epoch on our 253 validation paintings, keeps the best epoch by val CIDEr, saves it to
Drive, and scores it once on the 1,082 test paintings.

## Settings

In [ ]:
SMOKE_TEST = True      # True: 64 train / 32 val paintings, 1 epoch. False: the real run.
EPOCHS = 3             # the first replication scored best after epoch 0, so a few epochs is plenty
DRIVE_DIR = "/content/drive/MyDrive/KALE_files"
MS_MODEL = "iic/mplug_backbone_large_en"
RUN = 'sft_smoke' if SMOKE_TEST else 'sft_full'
OUT = f'/content/kale_out/{RUN}'                 # checkpoints are several GB: keep them on local disk
RESULTS_DRIVE = f'{DRIVE_DIR}/results_{RUN}'
SFT_CKPT_DRIVE = f'{DRIVE_DIR}/kale_sft_best{"_smoke" if SMOKE_TEST else ""}.pth'   # what the DPO notebook loads

Mount Drive, get KALE and our repo, check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob, json, shutil, re
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists('/content/KALE'):
    !git clone -q https://github.com/Yanbei-Jiang/Artwork-Interpretation.git /content/KALE
# Our repo: kale_dpo.py and the judge files (train/val split + preference pairs)
if not os.path.exists('/content/emm'):
    !git clone -q https://github.com/emmadionne1/10623_Project_EMMbeddings.git /content/emm
else:
    !git -C /content/emm pull -q
# Quick fixes without pushing: put an edited kale_dpo.py in KALE_files/ on Drive and it is used instead
if os.path.exists(f'{DRIVE_DIR}/kale_dpo.py'):
    shutil.copy(f'{DRIVE_DIR}/kale_dpo.py', '/content/emm/baselines/kale_dpo.py'); print('using kale_dpo.py from Drive')
DPO = '/content/emm/baselines/kale_dpo.py'
assert os.path.exists(DPO), 'kale_dpo.py not found: push it to GitHub or copy it to KALE_files/ on Drive'
%cd /content/KALE
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
if 'L4' in gpu:
    print('WARNING: PyTorch 1.11 may not support the L4. Use an A100 (real runs) or a T4 (smoke test).')

## Python 3.9 environment for KALE

In [ ]:
PY = '/content/kale-env/bin/python'
!pip install -q uv
if not os.path.exists(PY):
    !uv venv /content/kale-env --python 3.9.16
!uv pip install -q --python {PY} torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!uv pip install -q --python {PY} numpy==1.23.5 "transformers==4.18.0" pyyaml ruamel.yaml wandb oss2 ftfy regex tqdm six more-itertools "opencv-python-headless<4.9" timm==0.4.12 gdown pycocoevalcap
!uv pip install -q --python {PY} git+https://github.com/bckim92/language-evaluation.git
!apt-get install -y -qq default-jre-headless > /dev/null
!MPLBACKEND=Agg {PY} -c "import language_evaluation; language_evaluation.download('coco'); print('language_evaluation ok')"
!{PY} -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)); print(torch.ones(2).cuda()*2)"
RUNPY = f'MPLBACKEND=Agg WANDB_MODE=offline {PY}'   # prefix for every KALE-env command

##Data and image encoder

In [ ]:
assert os.path.getsize(f'{DRIVE_DIR}/ViT-L-14.tar') > 500e6, 'ViT-L-14.tar missing: run the replication notebook step 4c once'
!ln -sf {DRIVE_DIR}/ViT-L-14.tar /content/KALE/
!unzip -o -q $(find {DRIVE_DIR}/kale_shared -name data.zip | head -1) 'data/ArtCap/*' -d /content/KALE
if not os.path.exists('data/ArtCap/images'):
    !unzip -o -q $(find {DRIVE_DIR}/artcap_download -name images.zip | head -1) -d /content/KALE/data/ArtCap
train_val = json.load(open('data/ArtCap/artcap_train_val.json'))
test = json.load(open('data/ArtCap/artcap_test.json'))
missing = [e[0] for e in train_val + test if not os.path.exists(os.path.join('data/ArtCap/images', e[0]))]
print('train_val:', len(train_val), ' test:', len(test), ' (expected 2524 and 1082) | missing images:', len(missing))
assert not missing

# Our 90/10 split of KALE's train_val (seed 10623), recorded in the judge files
split_of = {str(json.loads(l)['image_id']): json.loads(l)['split'] for l in open('/content/emm/data/preferences/judgments.jsonl')}
img_id = lambda fn: os.path.splitext(os.path.basename(fn))[0]
train_entries = [e for e in train_val if split_of.get(img_id(e[0])) == 'train']
val_entries = [e for e in train_val if split_of.get(img_id(e[0])) == 'val']
print('our split: train', len(train_entries), ' val', len(val_entries), ' (expected 2271 and 253)')
assert len(train_entries) + len(val_entries) == len(train_val)

## Starting checkpoint
Same conversion as the replication notebook, including the `fusion_encoder` rename that KALE's loader needs.

In [ ]:
!pip install -q modelscope
from modelscope import snapshot_download
ms_dir = snapshot_download(MS_MODEL, cache_dir=DRIVE_DIR + '/modelscope')
CKPT = '/content/' + MS_MODEL.split('/')[-1] + '_kale.pth'
convert = f'''
import torch, glob, os
ckpt, ms_dir = {CKPT!r}, {ms_dir!r}
try:
    torch.load(ckpt, map_location='cpu'); print('converted checkpoint already there'); raise SystemExit
except (FileNotFoundError, RuntimeError, EOFError):
    pass
weights = [p for p in glob.glob(ms_dir + '/**/*', recursive=True)
           if p.endswith(('.bin', '.pth', '.pt')) and 'ViT' not in os.path.basename(p)]
f = max(weights, key=os.path.getsize)
print('source:', f, round(os.path.getsize(f) / 1e9, 2), 'GB')
sd = torch.load(f, map_location='cpu')
if 'model' in sd or 'module' in sd:
    sd = sd.get('model', sd.get('module'))
for prefix in ('module.', 'model.'):
    if all(k.startswith(prefix) for k in sd):
        sd = {{k[len(prefix):]: v for k, v in sd.items()}}
sd = {{('fusion_encoder.fusion.' + k[len('fusion_encoder.'):]
        if k.startswith('fusion_encoder.') and not k.startswith('fusion_encoder.fusion.') else k): v
      for k, v in sd.items()}}
torch.save({{'model': sd}}, ckpt)
print('saved', ckpt, len(sd), 'tensors')
'''
open('/content/convert_ckpt.py', 'w').write(convert)
!{PY} /content/convert_ckpt.py

## Point KALE at our split and configure the run
KALE trains on `artcap_train_val.json` and scores `artcap_test.json` after every epoch. We write
`split_train.json` (our train paintings) and `split_val.json` (our val paintings) and make KALE's
dataset code read those instead, so its per-epoch captions are validation captions. The original
annotation files stay untouched, and the test set is only used once for scoring.

In [ ]:
import yaml
tr, va = (train_entries[:64], val_entries[:32]) if SMOKE_TEST else (train_entries, val_entries)
json.dump(tr, open('data/ArtCap/split_train.json', 'w'))
json.dump(va, open('data/ArtCap/split_val.json', 'w'))

src_path = 'dataset/__init__.py'
if not os.path.exists(src_path + '.orig'):
    shutil.copy(src_path, src_path + '.orig')
src = open(src_path + '.orig').read()
a = "artcap_dataset(['./data/ArtCap/artcap_train_val.json'], train_transform"
b = "artcap_dataset(['./data/ArtCap/artcap_test.json'], test_transform"
assert src.count(a) == 1 and src.count(b) == 1, 'KALE dataset code changed: check dataset/__init__.py'
src = src.replace(a, a.replace('artcap_train_val', 'split_train')).replace(b, b.replace('artcap_test', 'split_val'))
open(src_path, 'w').write(src)

# KALE's own scorer crashes on METEOR (BrokenPipeError) and would stop training after epoch 0.
# Its result is only logged to wandb, so make the call non-fatal; we score in step 7.
s = open('KALE_artcap.py').read()
if 'cal_metric skipped' not in s:
    old = '            result = cal_metric(result_file)'
    assert old in s
    s = s.replace(old, '            try:\n                result = cal_metric(result_file)\n'
                       '            except Exception as e:\n                print("cal_metric skipped:", repr(e))')
    open('KALE_artcap.py', 'w').write(s)

cfg_path = 'configs/config.yaml'
if not os.path.exists(cfg_path + '.orig'):
    shutil.copy(cfg_path, cfg_path + '.orig')
cfg = yaml.load(open(cfg_path + '.orig'), Loader=yaml.Loader)
BATCH = 16 if 'A100' in gpu else 4
ACCUM = max(1, 512 // BATCH)          # same effective batch as KALE (256 x 2)
cfg['batch_size_train'] = BATCH
cfg['batch_size_test'] = BATCH
cfg['schedular']['epochs'] = 1 if SMOKE_TEST else EPOCHS
yaml.dump(cfg, open(cfg_path, 'w'))
os.makedirs(OUT, exist_ok=True); os.makedirs(RESULTS_DRIVE, exist_ok=True)
print(f'train {len(tr)} / val {len(va)} paintings | batch={BATCH} accum={ACCUM} epochs={cfg["schedular"]["epochs"]} | out={OUT}')

## Train and Score 


In [ ]:
%cd /content/KALE
!{RUNPY} KALE_artcap.py --checkpoint {CKPT} --output_dir {OUT} --accum_steps {ACCUM} 2>&1 | tee {OUT}/train_output.txt | awk '!/Train Epoch/ || /\[ *[0-9]*00\//' 
!grep -A1 "_IncompatibleKeys" {OUT}/train_output.txt | head -c 300; echo
!cp {OUT}/log.txt {OUT}/train_output.txt {OUT}/result/result_epoch*[0-9].json {RESULTS_DRIVE}/

In [ ]:
!pip install -q pycocoevalcap
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import pandas as pd

def score_file(path, meteor=False):
    """Score a KALE-format caption file against its own gold captions (x100), like helpers/metrics.py."""
    res = json.load(open(path))
    tok = PTBTokenizer()
    gen = tok.tokenize({i: [{'caption': ' '.join(str(r['pred_caption']).split()) or 'none'}] for i, r in enumerate(res)})
    ref = tok.tokenize({i: [{'caption': c} for c in r['gold_caption']] for i, r in enumerate(res)})
    out = {}
    b, _ = Bleu(4).compute_score(ref, gen, verbose=0)
    out.update({f'B{n}': v for n, v in enumerate(b, 1)})
    if meteor:
        try:
            m = Meteor().compute_score(ref, gen); out['METEOR'] = m[0] if isinstance(m, tuple) else m
        except Exception as e:
            print('METEOR failed (compute it later with helpers/metrics.py):', repr(e)[:120]); out['METEOR'] = float('nan')
    out['ROUGE_L'], _ = Rouge().compute_score(ref, gen)
    out['CIDEr'], _ = Cider().compute_score(ref, gen)
    out = {k: round(100 * v, 2) for k, v in out.items()}
    out['n'] = len(res); out['mean_len'] = round(sum(len(str(r['pred_caption']).split()) for r in res) / len(res), 1)
    return out

files = sorted(glob.glob(f'{OUT}/result/result_epoch*[0-9].json'), key=lambda f: int(re.findall(r'epoch(\d+)', f)[0]))
val_table = pd.DataFrame({int(re.findall(r'epoch(\d+)', f)[0]): score_file(f) for f in files}).T
val_table.index.name = 'epoch'
val_table.to_csv(f'{RESULTS_DRIVE}/val_scores_by_epoch.csv')
print(val_table.to_string())
BEST_EPOCH = int(val_table['CIDEr'].idxmax())
print(f'\nbest epoch by val CIDEr: {BEST_EPOCH} ({val_table.loc[BEST_EPOCH, "CIDEr"]})')

## Save to Drive
Saves only the model weights of the best epoch to drive. It is written locally first and then copied,
so a disconnect can't leave a half-written file on Drive.

In [ ]:
BEST_LOCAL = f'/content/kale_sft_best.pth'
save = f'''
import torch
ck = torch.load("{OUT}/checkpoint_{BEST_EPOCH:02d}.pth", map_location="cpu")
torch.save({{"model": ck["model"], "epoch": ck["epoch"], "source": "KALE retrain on our train split"}}, "{BEST_LOCAL}")
print("saved epoch", ck["epoch"])
'''
open('/content/save_best.py', 'w').write(save)
!{PY} /content/save_best.py
!cp {BEST_LOCAL} {SFT_CKPT_DRIVE}
assert os.path.getsize(SFT_CKPT_DRIVE) == os.path.getsize(BEST_LOCAL), 'Drive copy is incomplete: rerun this cell'
print('on Drive:', SFT_CKPT_DRIVE, round(os.path.getsize(SFT_CKPT_DRIVE) / 1e9, 2), 'GB')
# Free local disk: the epoch checkpoints include optimizer state (several GB each)
for f in glob.glob(f'{OUT}/checkpoint_*.pth'): os.remove(f)

In [ ]:
%cd /content/KALE
TEST_OUT = f'{OUT}/test_captions_sft.json'
smoke = '--smoke 32' if SMOKE_TEST else ''
EVAL_BATCH = 32 if 'A100' in gpu else 8
!{RUNPY} {DPO} generate --checkpoint {BEST_LOCAL} --split test --out {TEST_OUT} --eval_batch {EVAL_BATCH} {smoke}
test_scores = score_file(TEST_OUT, meteor=True)
json.dump(test_scores, open(f'{OUT}/test_scores_sft.json', 'w'), indent=1)
!cp {TEST_OUT} {OUT}/test_scores_sft.json {RESULTS_DRIVE}/
print(pd.Series(test_scores).to_string())